In [0]:
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("ingestion_date", "")

storage_account = dbutils.widgets.get("storage_account")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


def assert_row_count_gt_zero(df: DataFrame, df_name: str) -> None:
    row_count = df.count()
    if row_count == 0:
        raise ValueError(f"{df_name} is empty")
    print(f"[DQ PASS] {df_name} row count = {row_count}")


def assert_no_nulls(df: DataFrame, column_name: str, df_name: str) -> None:
    null_count = df.filter(F.col(column_name).isNull()).count()
    if null_count > 0:
        raise ValueError(f"{df_name}.{column_name} contains {null_count} null values")
    print(f"[DQ PASS] {df_name}.{column_name} contains no nulls")


def assert_unique_key(df: DataFrame, column_name: str, df_name: str) -> None:
    total_count = df.count()
    distinct_count = df.select(column_name).distinct().count()
    if total_count != distinct_count:
        raise ValueError(
            f"{df_name}.{column_name} is not unique: total_count={total_count}, distinct_count={distinct_count}"
        )
    print(f"[DQ PASS] {df_name}.{column_name} is unique")


def assert_non_negative(df: DataFrame, column_name: str, df_name: str) -> None:
    bad_count = df.filter(F.col(column_name) < 0).count()
    if bad_count > 0:
        raise ValueError(f"{df_name}.{column_name} contains {bad_count} negative values")
    print(f"[DQ PASS] {df_name}.{column_name} contains no negative values")


def assert_referential_integrity(
    child_df: DataFrame,
    child_column: str,
    parent_df: DataFrame,
    parent_column: str,
    relationship_name: str,
) -> None:
    missing_df = (
        child_df.select(child_column).distinct()
        .join(
            parent_df.select(parent_column).distinct(),
            child_df[child_column] == parent_df[parent_column],
            "left_anti",
        )
    )
    missing_count = missing_df.count()
    if missing_count > 0:
        raise ValueError(
            f"Referential integrity failed for {relationship_name}: {missing_count} unmatched keys"
        )
    print(f"[DQ PASS] Referential integrity passed for {relationship_name}")

In [0]:
products_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/dummyjson/products/ingestion_date={ingestion_date}/products.json"
users_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/dummyjson/users/ingestion_date={ingestion_date}/users.json"
carts_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/dummyjson/carts/ingestion_date={ingestion_date}/carts.json"

In [0]:
silver_products_path = (
    f"abfss://silver@{storage_account}.dfs.core.windows.net/"
    f"dummyjson/products/ingestion_date={ingestion_date}/"
)

silver_customers_path = (
    f"abfss://silver@{storage_account}.dfs.core.windows.net/"
    f"dummyjson/customers/ingestion_date={ingestion_date}/"
)

silver_orders_path = (
    f"abfss://silver@{storage_account}.dfs.core.windows.net/"
    f"dummyjson/orders/ingestion_date={ingestion_date}/"
)

silver_order_items_path = (
    f"abfss://silver@{storage_account}.dfs.core.windows.net/"
    f"dummyjson/order_items/ingestion_date={ingestion_date}/"
)

In [0]:
gold_products_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/dummyjson/dim_products/"
gold_customers_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/dummyjson/dim_customers/"
gold_orders_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/dummyjson/fact_orders/"
gold_order_items_path = f"abfss://gold@{storage_account}.dfs.core.windows.net/dummyjson/fact_order_items/"

In [0]:
silver_products = spark.read.parquet(silver_products_path)
silver_customers = spark.read.parquet(silver_customers_path)
silver_orders = spark.read.parquet(silver_orders_path)
silver_order_items = spark.read.parquet(silver_order_items_path)

display(silver_products)
display(silver_customers)
display(silver_orders)
display(silver_order_items)

In [0]:
dim_products = silver_products.dropDuplicates(["product_id"])
dim_products = dim_products.withColumn("ingestion_date", F.lit(ingestion_date))

assert_row_count_gt_zero(dim_products, "dim_products")
assert_no_nulls(dim_products, "product_id", "dim_products")
assert_unique_key(dim_products, "product_id", "dim_products")

dim_customers = silver_customers.dropDuplicates(["customer_id"])
dim_customers = dim_customers.withColumn("ingestion_date", F.lit(ingestion_date))

assert_row_count_gt_zero(dim_customers, "dim_customers")
assert_no_nulls(dim_customers, "customer_id", "dim_customers")
assert_unique_key(dim_customers, "customer_id", "dim_customers")

In [0]:
fact_orders = silver_orders.select(
    "order_id",
    "customer_id",
    "order_total",
    "discounted_order_total",
    "total_products",
    "total_quantity"
)
fact_orders = fact_orders.withColumn("ingestion_date", F.lit(ingestion_date))

assert_row_count_gt_zero(fact_orders, "fact_orders")
assert_no_nulls(fact_orders, "order_id", "fact_orders")
assert_unique_key(fact_orders, "order_id", "fact_orders")
assert_no_nulls(fact_orders, "customer_id", "fact_orders")
assert_non_negative(fact_orders, "order_total", "fact_orders")
assert_non_negative(fact_orders, "discounted_order_total", "fact_orders")

fact_order_items = silver_order_items.select(
    "order_item_id",
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "line_total",
    "discount_percentage",
    "discounted_line_total"
)
fact_order_items = fact_order_items.withColumn("ingestion_date", F.lit(ingestion_date))

assert_row_count_gt_zero(fact_order_items, "fact_order_items")
assert_no_nulls(fact_order_items, "order_item_id", "fact_order_items")
assert_unique_key(fact_order_items, "order_item_id", "fact_order_items")
assert_no_nulls(fact_order_items, "order_id", "fact_order_items")
assert_no_nulls(fact_order_items, "product_id", "fact_order_items")
assert_non_negative(fact_order_items, "quantity", "fact_order_items")
assert_non_negative(fact_order_items, "unit_price", "fact_order_items")
assert_non_negative(fact_order_items, "line_total", "fact_order_items")

In [0]:
assert_referential_integrity(
    child_df=fact_orders,
    child_column="customer_id",
    parent_df=dim_customers,
    parent_column="customer_id",
    relationship_name="fact_orders.customer_id -> dim_customers.customer_id",
)

assert_referential_integrity(
    child_df=fact_order_items,
    child_column="product_id",
    parent_df=dim_products,
    parent_column="product_id",
    relationship_name="fact_order_items.product_id -> dim_products.product_id",
)

assert_referential_integrity(
    child_df=fact_order_items,
    child_column="order_id",
    parent_df=fact_orders,
    parent_column="order_id",
    relationship_name="fact_order_items.order_id -> fact_orders.order_id",
)

In [0]:
dim_products.write.mode("overwrite").parquet(gold_products_path)
dim_customers.write.mode("overwrite").parquet(gold_customers_path)
fact_orders.write.mode("overwrite").parquet(gold_orders_path)
fact_order_items.write.mode("overwrite").parquet(gold_order_items_path)

display(spark.read.parquet(gold_products_path))
display(spark.read.parquet(gold_customers_path))
display(spark.read.parquet(gold_orders_path))
display(spark.read.parquet(gold_order_items_path))

In [0]:
dim_products = spark.read.parquet(gold_products_path)
dim_customers = spark.read.parquet(gold_customers_path)
fact_orders = spark.read.parquet(gold_orders_path)
fact_order_items = spark.read.parquet(gold_order_items_path)

display(dim_products)

In [0]:
snowflake_options = {
    "sfURL": "<URL>>",
    "sfUser": "<USER>>",
    "sfPassword": "<PASSWORD>",
    "sfDatabase": "FAKESTORE_DB",
    "sfSchema": "GOLD",
    "sfWarehouse": "COMPUTE_WH",
    "sfRole": "ACCOUNTADMIN"
}

In [0]:
dim_products_sf = dim_products.select(
    "product_id",
    "title",
    "description",
    "category",
    "price",
    "brand",
    "sku",
    "weight",
    "warranty_information",
    "shipping_information",
    "availability_status",
    "return_policy",
    "minimum_order_quantity",
    "rating",
    "stock",
    "thumbnail_url",
    "ingestion_date"
)

display(dim_products_sf)

In [0]:
dim_products_sf.write \
    .format("snowflake") \
    .options(**snowflake_options) \
    .option("dbtable", "DIM_PRODUCTS") \
    .mode("overwrite") \
    .save()

In [0]:
dim_customers_sf = dim_customers.select(
    "customer_id",
    "first_name",
    "last_name",
    "maiden_name",
    "age",
    "gender",
    "email",
    "phone",
    "username",
    "birth_date",
    "blood_group",
    "height_cm",
    "weight_kg",
    "eye_color",
    "city",
    "state",
    "state_code",
    "postal_code",
    "country",
    "latitude",
    "longitude",
    "ingestion_date"
)

dim_customers_sf.write \
    .format("snowflake") \
    .options(**snowflake_options) \
    .option("dbtable", "DIM_CUSTOMERS") \
    .mode("overwrite") \
    .save()

In [0]:
fact_orders_sf = fact_orders.select(
    "order_id",
    "customer_id",
    "order_total",
    "discounted_order_total",
    "total_products",
    "total_quantity",
    "ingestion_date"
)

fact_orders_sf.write \
    .format("snowflake") \
    .options(**snowflake_options) \
    .option("dbtable", "FACT_ORDERS") \
    .mode("overwrite") \
    .save()

In [0]:
fact_order_items_sf = fact_order_items.select(
    "order_item_id",
    "order_id",
    "product_id",
    "quantity",
    "unit_price",
    "line_total",
    "discount_percentage",
    "discounted_line_total",
    "ingestion_date"
)

fact_order_items_sf.write \
    .format("snowflake") \
    .options(**snowflake_options) \
    .option("dbtable", "FACT_ORDER_ITEMS") \
    .mode("overwrite") \
    .save()